# LTX Video Test: Text-to-Video + Image-to-Video

Prueba para verificar [Lightricks/LTX-Video](https://huggingface.co/Lightricks/LTX-Video).

**Problema conocido:** el modelo completo necesita >12 GB RAM durante la carga porque PyTorch lee cada shard completo a RAM antes de mandarlo a GPU.

**Soluciones:**
- Opcion A: `device_map="balanced"` con accelerate (carga directo a GPU y libera RAM)
- Opcion B: modelo destilado (solo ~4 GB RAM)

In [ ]:
# ---- 1. INSTALAR DEPENDENCIAS ----
import os, torch
!pip install -qU diffusers transformers accelerate sentencepiece
!pip install -q imageio[ffmpeg] pillow psutil

In [ ]:
# ---- 2. IMPORTS + DIAGNOSTICO ----
import torch, gc, psutil
from diffusers import LTXPipeline, LTXConditionPipeline
from diffusers.utils import load_image, export_to_video
from diffusers.pipelines.ltx.pipeline_ltx_condition import LTXVideoCondition
from IPython.display import Video

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
DTYPE = torch.bfloat16

ram_gb = psutil.virtual_memory().total / 1024**3
ram_avail = psutil.virtual_memory().available / 1024**3
print(f'RAM total: {ram_gb:.1f} GB, disponible: {ram_avail:.1f} GB')
print(f'Device: {DEVICE}')
if DEVICE == 'cuda':
    gpu = torch.cuda.get_device_name()
    vram = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print(f'GPU: {gpu} ({vram:.1f} GB VRAM)')

## Opcion A: Modelo completo Lightricks/LTX-Video

Usa `device_map="balanced"` para que accelerate cargue cada componente directo a GPU y libere la copia en RAM. El pico de RAM se limita al shard mas grande (~2 GB) en lugar del modelo completo (~10 GB).

**Nota:** Si tienes <14 GB RAM, usa la opcion B.

In [ ]:
# ---- 3A. TEXT-TO-VIDEO (device_map="balanced") ----

gc.collect(); torch.cuda.empty_cache()

ram_ok = psutil.virtual_memory().available / 1024**3 > 3
if not ram_ok:
    print('ERROR: memoria disponible insuficiente. Usa Opcion B.')
else:
    print('Cargando LTXPipeline con device_map="balanced"...')
    pipe = LTXPipeline.from_pretrained(
        'Lightricks/LTX-Video',
        torch_dtype=DTYPE,
        device_map='balanced',
    )
    pipe.vae.enable_tiling()
    print('OK')

    prompt = 'A cinematic drone shot of a misty mountain range at sunrise.'

    with torch.inference_mode():
        output = pipe(
            prompt=prompt,
            width=640, height=480,
            num_frames=33,
            num_inference_steps=30,
            guidance_scale=5.0,
        )

    export_to_video(output.frames[0], '/content/ltx_t2v.mp4', fps=24)
    print('Video: /content/ltx_t2v.mp4')
    Video('/content/ltx_t2v.mp4', embed=True, width=512)

    del pipe; gc.collect(); torch.cuda.empty_cache()

In [ ]:
# ---- 4A. IMAGE-TO-VIDEO (device_map="balanced") ----

gc.collect(); torch.cuda.empty_cache()

if psutil.virtual_memory().available / 1024**3 < 3:
    print('ERROR: memoria insuficiente. Usa Opcion B.')
else:
    print('Cargando LTXConditionPipeline...')
    pipe = LTXConditionPipeline.from_pretrained(
        'Lightricks/LTX-Video',
        torch_dtype=DTYPE,
        device_map='balanced',
    )
    pipe.vae.enable_tiling()

    image = load_image('https://huggingface.co/datasets/huggingface/documentation-images/'
                        'resolve/main/diffusers/guitar-man.png')
    condition = LTXVideoCondition(image=image, frame_index=0)

    with torch.inference_mode():
        output = pipe(
            prompt='A man playing guitar, slow motion.',
            conditions=[condition],
            width=640, height=480,
            num_frames=33,
            num_inference_steps=30,
            guidance_scale=5.0,
        )

    export_to_video(output.frames[0], '/content/ltx_i2v.mp4', fps=24)
    print('Video: /content/ltx_i2v.mp4')
    Video('/content/ltx_i2v.mp4', embed=True, width=512)

    del pipe; gc.collect(); torch.cuda.empty_cache()

## Opcion B: LTX-2.3 Distilled (8 steps, RAM baja)

Modelo destilado de ~2B que funciona con ~4 GB RAM y ~6 GB VRAM. 8 steps de inferencia, genera video + audio sincronizado.

In [ ]:
# ---- 3B. LTX 2.3 DISTILLED ----

!pip install -qU git+https://github.com/huggingface/diffusers.git

gc.collect(); torch.cuda.empty_cache()

from diffusers import LTX2Pipeline
from diffusers.utils import export_to_video

print('Cargando LTX-2.3 Distilled...')
pipe = LTX2Pipeline.from_pretrained(
    'diffusers/LTX-2.3-Distilled-Diffusers',
    torch_dtype=torch.bfloat16,
)
pipe.to('cuda')

prompt = 'A flowing river in a forest at golden hour, gentle wind in the leaves.'
print('Generando (8 steps, 768x512, 49 frames)...')

video, audio = pipe(
    prompt=prompt,
    width=768, height=512,
    num_frames=49,
    frame_rate=24,
    num_inference_steps=8,
    guidance_scale=1.0,
    output_type='np',
    return_dict=False,
)

export_to_video(video[0], '/content/ltx_fast.mp4', fps=24)
print('Video: /content/ltx_fast.mp4')
Video('/content/ltx_fast.mp4', embed=True, width=512)

del pipe; gc.collect(); torch.cuda.empty_cache()

In [ ]:
# ---- 5. LIMPIEZA ----
gc.collect(); torch.cuda.empty_cache()
print('Done')